# Making Batches of Data to a Folder

Note that this will save data both into our lab-wide Tiled server and into our local folder

Here we use the ophyd-async devices

You'll need to have TILED_API_KEY and TILED_URI environment variables set.

## Imports

### Global Imports

In [1]:
import numpy as np

from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from bluesky.plans import count, list_scan
import bluesky.plan_stubs as bps

from bluesky.callbacks.tiled_writer import TiledWriter
from tiled.server import SimpleTiledServer
from tiled.client import from_uri

from ophyd_async.core import init_devices

import matplotlib.pyplot as plt

### Local-File Imports

In [2]:
from sidekick_model3_CA_devices_v4 import PulseGenerator, LilLaser, Diode
from sidekick_model3_PVA_devices_v4 import DiodePVA
from array_steps import array_one_nd_step_with_reps
from sidekick_model3_helpers import condition_pulse

## Initialize our Sidekick Model 3 Devices

#### Start Bluesky RunEngine (before initializing ophyd-async devices)

In [3]:
RE = RunEngine()

#### Initialize ophyd-async devices

In [4]:
with init_devices():
    pulsegen = PulseGenerator("PULSEGEN:", name="pulsegen")
    laser = LilLaser("LASER:", name="laser")
    electron = Diode("ELECTRON:", name="electron")
    proton = Diode("PROTON:", name="proton")
    electron_pva = DiodePVA(prefix="pva://ELECTRON-DAQ:", name="electron_pva")
    proton_pva = DiodePVA(prefix="pva://PROTON-DAQ:", name="proton_pva")

## Connect to our Lab-Wide Tiled Server, and Subscribe to Run Engine

In [5]:
import os
from tiled.client import from_uri

def get_tiled_client():
    """Connect to Tiled using environment variables.
    Simple helper function written by ChatGPT"""

    tiled_uri = os.environ.get("TILED_URI")
    tiled_api_key = os.environ.get("TILED_API_KEY")

    if tiled_uri is None:
        raise RuntimeError(
            "Missing TILED_URI.\n"
            "Example:\n"
            "    export TILED_URI='http://your-tiled-server.lan:8000'\n"
        )

    if tiled_api_key is None:
        raise RuntimeError(
            "Missing TILED_API_KEY.\n"
            "Example:\n"
            "    export TILED_API_KEY='your-api-key-here'\n"
        )

    return from_uri(tiled_uri, api_key=tiled_api_key)

In [6]:
tiled_client = get_tiled_client()

In [7]:
tw = TiledWriter(tiled_client)
RE.subscribe(tw)

0

## Create and Execute Plan to Put Sidekick in a Known Initial State

### Create Plan to Set timing settings and rep-rate.

In [8]:
# Written by ChatGPT with help from Scott Feister on 2026-06-22.
# Simple Bluesky pre-run setup helper for putting Sidekick devices
# into a known initial state before opening a run.

def prepare_for_run():
    """Put the Sidekick Model 3 into the standard initial state before a run.
    """

    # Set all delta-t values.
    yield from bps.mv(
        proton.dt, 5.0e-6,        # seconds
        electron.dt, 5.0e-6,      # seconds
        laser.powers_dt, 5.0,     # microseconds
    )

    # Set trigger delays to known values.
    yield from bps.mv(
        pulsegen.ch2_delay, 100.0,    # microseconds; proton delay
        pulsegen.ch3_delay, 100.0,    # microseconds; electron delay
        pulsegen.ch4_delay, 100.0,    # microseconds; laser delay
    )

    # Set system repetition rate.
    yield from bps.mv(
        pulsegen.reprate, 10.0,       # Hz
    )

### Execute this Plan

In [9]:
RE(prepare_for_run())

()

## 50 random traces with 3 reps on each

### Export data

In [12]:
"""
Export one specific kind of run to from Tiled to HDF5.

Created by ChatGPT with help from Scott Feister on 2026-06-25.
"""

from pathlib import Path

import h5py
import numpy as np


def export_run_to_hdf5_custom2(tiled_client, uid, filename=None):
    if filename is None:
        filename = f"run_{uid}.h5"

    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    primary = tiled_client[uid, "primary"]

    with h5py.File(filename, "w") as f:
        f.attrs["uid"] = uid

        g = f.create_group("primary")

        g.create_dataset("scan_step", data=np.asarray(primary["scan_step"]))
        g.create_dataset("scan_rep", data=np.asarray(primary["scan_rep"]))
        g.create_dataset("laser_powers", data=np.asarray(primary["laser-powers-readback"]))
        g.create_dataset("electron_trace", data=np.asarray(primary["electron_pva-trace-array"]))
        g.create_dataset("electron_shot_num", data=np.asarray(primary["electron_pva-trace-uniqueId"]))
        g.create_dataset("proton_trace", data=np.asarray(primary["proton_pva-trace-array"]))
        g.create_dataset("proton_shot_num", data=np.asarray(primary["proton_pva-trace-uniqueId"]))

    return filename

In [16]:
#!/usr/bin/env python3
"""
export_batch_hdf5.py

Export one Tiled run to the next numbered HDF5 batch file.

Each exported run becomes one file:

    outputs/batches/batch_001.h5
    outputs/batches/batch_002.h5
    ...

Written by ChatGPT with help from Scott Feister on 2026-06-26.
"""

from pathlib import Path
import re


# -----------------------
# Settings
# -----------------------

outputs_dir = Path("outputs")
batches_dir = outputs_dir / "batches"


# -----------------------
# Helper: find next batch number
# -----------------------

def get_existing_batch_numbers():
    batch_numbers = []

    for path in batches_dir.glob("batch_*"):
        # Accept both old-style folders:
        #     batch_001/
        #
        # and new-style HDF5 files:
        #     batch_001.h5
        match = re.fullmatch(r"batch_(\d+)(?:\.h5)?", path.name)

        if match:
            batch_numbers.append(int(match.group(1)))

    return sorted(batch_numbers)


def get_next_batch_number():
    existing_numbers = get_existing_batch_numbers()

    if not existing_numbers:
        return 1

    return max(existing_numbers) + 1


def get_next_batch_hdf5_path():
    batches_dir.mkdir(parents=True, exist_ok=True)

    batch_number = get_next_batch_number()
    batch_name = f"batch_{batch_number:03d}.h5"

    return batches_dir / batch_name


# -----------------------
# Export one run
# -----------------------

def export_run_as_next_batch(tiled_client, uid):
    batch_path = get_next_batch_hdf5_path()

    export_run_to_hdf5_custom2(
        tiled_client,
        uid,
        batch_path,
    )

    print(f"Wrote {batch_path}")

    return batch_path

In [17]:
export_run_as_next_batch(tiled_client, uid)

Wrote outputs\batches\batch_018.h5


WindowsPath('outputs/batches/batch_018.h5')

## Run and Save - in a Loop

Note that this has failed before because of a single rogue waveform from laser_powers coming into Tiled as 9 elements long.

In [18]:
npulses = 50
reps = 3
detectors = [electron_pva.trace, proton_pva.trace]
motor = laser.powers

for i in range(3):
    pulse_list = [condition_pulse(np.round(np.random.rand(100)*255)) for i in range(npulses)]
    motor_points = pulse_list
    md = {"user_note" : "Acquire and save in a loop, repeated list scans on Sidekick Model 3", "outer_loop_i": i}
    uid, = RE(list_scan(detectors, motor, motor_points, md=md, per_step=array_one_nd_step_with_reps(reps=reps)))
    export_run_as_next_batch(tiled_client, uid)

Wrote outputs\batches\batch_019.h5
Wrote outputs\batches\batch_020.h5
Wrote outputs\batches\batch_021.h5
Wrote outputs\batches\batch_022.h5
Wrote outputs\batches\batch_023.h5
Wrote outputs\batches\batch_024.h5
Wrote outputs\batches\batch_025.h5
Wrote outputs\batches\batch_026.h5
Wrote outputs\batches\batch_027.h5
Wrote outputs\batches\batch_028.h5
Wrote outputs\batches\batch_029.h5
Wrote outputs\batches\batch_030.h5
Wrote outputs\batches\batch_031.h5
Wrote outputs\batches\batch_032.h5
Wrote outputs\batches\batch_033.h5
Wrote outputs\batches\batch_034.h5
Wrote outputs\batches\batch_035.h5
Wrote outputs\batches\batch_036.h5
Wrote outputs\batches\batch_037.h5
Wrote outputs\batches\batch_038.h5
